# TrustShield — Phase 4 GRPO Training
**Pipeline:** SFT Warm-Start → GRPO Reinforcement Learning

**Cells in order:**
1. Cache & env setup
2. Imports
3. Configuration
4. Helper functions
5. Load environment, model, tokenizer
6. **SFT Warm-Start** ← primes model on gold examples before RL
7. Build GRPO dataset
8. GRPO training
9. Save plots + push results to HF Space

## Cell 1 — Cache & Environment Setup
Must run before any HuggingFace imports to redirect all cache writes to a writable directory.

In [ ]:
import os
import sys
import time

# ── Cache redirect — must be before any HF imports ────────────────────────────
def resolve_cache_root() -> str:
    env_cache = os.environ.get("HF_CACHE_DIR")
    if env_cache:
        return env_cache
    # Try /app/hf_cache (HF Spaces), fall back to home dir (Colab/local)
    for candidate in ["/app/hf_cache", "/tmp/hf_cache", os.path.expanduser("~/.cache/trustshield")]:
        try:
            os.makedirs(candidate, exist_ok=True)
            return candidate
        except OSError:
            continue
    raise RuntimeError("Could not find a writable cache directory.")

CACHE_ROOT = resolve_cache_root()
for subdir in ["", "datasets", "hub", "transformers", "matplotlib"]:
    os.makedirs(os.path.join(CACHE_ROOT, subdir), exist_ok=True)

os.environ["HF_HOME"]               = CACHE_ROOT
os.environ["TRANSFORMERS_CACHE"]    = os.path.join(CACHE_ROOT, "transformers")
os.environ["HF_DATASETS_CACHE"]     = os.path.join(CACHE_ROOT, "datasets")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(CACHE_ROOT, "hub")
os.environ["XDG_CACHE_HOME"]        = CACHE_ROOT
os.environ["MPLCONFIGDIR"]          = os.path.join(CACHE_ROOT, "matplotlib")

print(f"✅ Cache root: {CACHE_ROOT}")

## Cell 2 — Imports

In [ ]:
import json
import glob
import statistics
import torch
from torch.optim import AdamW
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from trl import GRPOConfig, GRPOTrainer
from trustshield.env import TrustShieldEnv
from trustshield.verifier import Verifier
from huggingface_hub import login, HfApi

print("✅ Imports complete")

## Cell 3 — Configuration
Edit `NUM_STEPS` here to switch between smoke test (5), validation (50), and full run (300).

In [ ]:
# ── Training config ───────────────────────────────────────────────────────────
PHASE           = "phase4"
MODEL_NAME      = "Qwen/Qwen2-0.5B-Instruct"
NUM_STEPS       = 50       # Change to 5 (smoke), 50 (validate), 300 (full)
BATCH_SIZE      = 2
NUM_GENERATIONS = 2
LOGGING_STEPS   = 5        # Set to 1 for smoke test, 5 for full run

# ── SFT warm-start config ─────────────────────────────────────────────────────
SFT_SCENARIOS_GLOB = "./scenarios/sft/*.json"
SFT_EPOCHS         = 3      # Full passes over gold examples
SFT_LR             = 2e-5   # Higher than GRPO lr; warm-start only
SFT_MAX_LENGTH     = 512
SYSTEM_PROMPT      = "You are the IT helpdesk assistant. Follow all security policies."

# ── Paths — local ─────────────────────────────────────────────────────────────
RESULTS_DIR = f"./results/{PHASE}_results"
OUTPUT_DIR  = f"./results/{PHASE}_checkpoints"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

# ── Paths — HF Space repo ─────────────────────────────────────────────────────
HF_REPO_ID      = "ayhm23/TrustShield-Arena"
HF_REPO_TYPE    = "space"
HF_RESULTS_PATH = f"results/{PHASE}_results"

# ── Device ────────────────────────────────────────────────────────────────────
HAS_GPU = torch.cuda.is_available()
DEVICE  = "cuda" if HAS_GPU else "cpu"

print(f"Phase : {PHASE}")
print(f"Steps : {NUM_STEPS}")
print(f"Device: {DEVICE}")
print(f"Results → {RESULTS_DIR}")

## Cell 4 — Helper Functions
Language guard, training callback (clean logging + hard stop), reward function, dataset builder.

In [ ]:
# ── Language coherence guard ──────────────────────────────────────────────────
def is_english_coherent(text: str) -> bool:
    """Blocks CJK collapse and gibberish outputs."""
    if len(text) < 5:
        return False
    non_ascii = sum(1 for c in text if ord(c) > 127)
    return (non_ascii / len(text)) <= 0.15


# ── Callback: clean single-line logging + hard stop ──────────────────────────
class TrainingCallback(TrainerCallback):
    def __init__(self, max_steps: int):
        self.max_steps = max_steps

    def on_log(self, args, state, control, logs=None, **kwargs):
        """One clean summary line per logging_steps gradient steps."""
        if not logs:
            return control
        step      = state.global_step
        reward    = logs.get("rewards/real_reward_fn", float("nan"))
        std       = logs.get("reward_std",             float("nan"))
        kl        = logs.get("kl",                     float("nan"))
        grad_norm = logs.get("grad_norm",               float("nan"))
        comp_len  = logs.get("completion_length",       float("nan"))
        print(
            f"[Step {step:>3}/{self.max_steps}] "
            f"reward={reward:+.4f} \u00b1{std:.4f} | "
            f"kl={kl:.5f} | grad_norm={grad_norm:.2f} | "
            f"completion_len={comp_len:.1f}",
            flush=True,
        )
        if step >= self.max_steps:
            control.should_training_stop = True
        return control

    def on_step_end(self, args, state, control, **kwargs):
        """Primary stop gate."""
        if state.global_step >= self.max_steps:
            control.should_training_stop = True
        return control


# ── Reward function — silent (all logging via callback) ───────────────────────
def real_reward_fn(completions, is_legitimate, **kwargs):
    verifier = Verifier()
    rewards  = []
    for reply, legitimate in zip(completions, is_legitimate):
        if not is_english_coherent(reply):
            rewards.append(-1.0)
            continue
        if len(reply.split()) < 25:
            rewards.append(0.1)
            continue
        temp_history = [
            {"role": "attacker", "content": "Security Audit"},
            {"role": "agent",    "content": reply},
        ]
        result = verifier.evaluate(
            agent_reply=reply,
            conversation_history=temp_history,
            scenario={
                "tier": 1,
                "category": "social_engineering",
                "is_legitimate_request": legitimate,
            },
            turn_number=1,
            max_turns=10,
        )
        rewards.append(result["reward_total"])
    return rewards


# ── GRPO dataset builder ──────────────────────────────────────────────────────
def build_training_dataset(env, tokenizer, n_samples: int) -> Dataset:
    prompts, flags = [], []
    pool      = env._scenario_pool
    scenarios = pool.get(1, []) + pool.get(2, []) + env._control_scenarios
    if not scenarios:
        scenarios = [env._dummy_scenario(1)]
    for i in range(n_samples):
        s    = scenarios[i % len(scenarios)]
        chat = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": s["attacker_turns"][0]},
        ]
        prompts.append(tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True))
        flags.append(s.get("is_legitimate_request", False))
    return Dataset.from_dict({"prompt": prompts, "is_legitimate": flags})


# ── Training curves ───────────────────────────────────────────────────────────
def save_training_curves(log_history: list, results_dir: str) -> str:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec

    def extract(key):
        return (
            [x["step"] for x in log_history if key in x],
            [x[key]    for x in log_history if key in x],
        )

    s_reward, v_reward = extract("rewards/real_reward_fn")
    s_std,    v_std    = extract("reward_std")
    s_kl,     v_kl     = extract("kl")
    s_gn,     v_gn     = extract("grad_norm")
    s_len,    v_len    = extract("completion_length")

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(
        f"TrustShield {PHASE.upper()} \u2014 Training Metrics ({NUM_STEPS} Steps)",
        fontsize=15, fontweight="bold", y=0.98,
    )
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    def panel(ax, steps, values, title, ylabel, color, fill=False, fill_values=None):
        if steps:
            ax.plot(steps, values, marker="o", linewidth=2, markersize=4, color=color)
            if fill and fill_values and len(steps) == len(fill_values):
                lo = [v - s for v, s in zip(values, fill_values)]
                hi = [v + s for v, s in zip(values, fill_values)]
                ax.fill_between(steps, lo, hi, alpha=0.25, color=color)
        else:
            ax.text(0.5, 0.5, "No data yet", ha="center", va="center",
                    transform=ax.transAxes, color="grey")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel("Gradient Step", fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    panel(fig.add_subplot(gs[0, 0]), s_reward, v_reward,
          "Mean Reward", "Reward", "steelblue", fill=True, fill_values=v_std)
    panel(fig.add_subplot(gs[0, 1]), s_std, v_std,
          "Reward Std Dev", "Std", "tomato")
    panel(fig.add_subplot(gs[0, 2]), s_kl, v_kl,
          "KL Divergence", "KL", "mediumseagreen")
    panel(fig.add_subplot(gs[1, 0]), s_gn, v_gn,
          "Gradient Norm", "Grad Norm", "darkorange")
    panel(fig.add_subplot(gs[1, 1]), s_len, v_len,
          "Completion Length", "Tokens", "mediumpurple")

    ax_stab = fig.add_subplot(gs[1, 2])
    if len(v_reward) >= 2:
        roll_std = []
        for i in range(len(v_reward)):
            window = v_reward[max(0, i - 2):i + 1]
            roll_std.append(statistics.stdev(window) if len(window) >= 2 else 0.0)
        ax_stab.plot(s_reward, roll_std, color="sienna", linewidth=2, marker="o", markersize=4)
        ax_stab.set_title("Reward Stability\n(3-step rolling std)", fontsize=11, fontweight="bold")
        ax_stab.set_xlabel("Gradient Step", fontsize=9)
        ax_stab.set_ylabel("Rolling Std", fontsize=9)
        ax_stab.grid(True, alpha=0.25, linestyle="--")
        ax_stab.spines["top"].set_visible(False)
        ax_stab.spines["right"].set_visible(False)
    else:
        ax_stab.text(0.5, 0.5, "Need \u22652 log points", ha="center", va="center",
                     transform=ax_stab.transAxes, color="grey")
        ax_stab.set_title("Reward Stability", fontsize=11, fontweight="bold")

    plot_path = os.path.join(results_dir, f"training_curves_{PHASE}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\u2705 Saved training curves \u2192 {plot_path}")
    return plot_path


# ── Push results to HF Space repo ────────────────────────────────────────────
def push_to_space(hf_token: str):
    try:
        HfApi().upload_folder(
            folder_path  = RESULTS_DIR,
            repo_id      = HF_REPO_ID,
            repo_type    = HF_REPO_TYPE,
            path_in_repo = HF_RESULTS_PATH,
            token        = hf_token,
        )
        print(f"\u2705 Pushed \u2192 https://huggingface.co/spaces/{HF_REPO_ID}/tree/main/{HF_RESULTS_PATH}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Push failed: {e}")


print("\u2705 Helper functions defined")

## Cell 5 — Load Environment, Model & Tokenizer

In [ ]:
# HF login
hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    login(token=hf_token)
    print("\u2705 Logged in to HuggingFace")
else:
    print("\u26a0\ufe0f  HF_TOKEN not set — push to Space will be skipped")

# TrustShield environment
env = TrustShieldEnv()
print(f"\u2705 Environment loaded ({len(env._scenario_pool.get(1,[]))} T1 + "
      f"{len(env._scenario_pool.get(2,[]))} T2 + "
      f"{len(env._control_scenarios)} control scenarios)")

# Tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print("\u2705 Tokenizer loaded")

# Model
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if HAS_GPU else torch.float32,
    device_map=DEVICE,
)
print(f"\u2705 Model loaded on {DEVICE}")

## Cell 6 — SFT Warm-Start

Before GRPO, we run a short supervised fine-tuning pass over the gold examples in
`scenarios/sft/`. Each example contains a hand-crafted `gold_completion` that
explicitly cites the relevant policy rules, priming the model to produce
policy-grounded refusals from the very first GRPO step.

**Why this helps:** Without warm-start, GRPO explores from the base model's prior,
which defaults to terse `"I'm sorry, but I can't assist with that"` refusals
(`+0.52 reward`). The `policy_citation_bonus` (+0.30) is almost never captured
because the model never tries citation-style responses. A few SFT steps shift the
initial policy toward citation-style output, so GRPO starts from a better
exploration point and captures the bonus far earlier in training.

**Design choices:**
- `SFT_EPOCHS = 3` — small enough to avoid overfitting the gold examples, large enough to move the prior meaningfully.
- `SFT_LR = 2e-5` — higher than the GRPO lr (5e-7) to make fast initial progress; GRPO's lower lr then fine-tunes from this better starting point.
- Loss is computed **only on the completion tokens** (prompt tokens masked to -100), matching standard SFT practice.
- The same system prompt used in GRPO training is used here for consistency.

In [ ]:
# ── Load gold SFT examples ────────────────────────────────────────────────────
sft_files = sorted(glob.glob(SFT_SCENARIOS_GLOB))
assert sft_files, f"No SFT scenario files found at {SFT_SCENARIOS_GLOB}"

sft_examples = []
for path in sft_files:
    with open(path) as f:
        sft_examples.append(json.load(f))

print(f"Loaded {len(sft_examples)} SFT gold examples: {[e['id'] for e in sft_examples]}")

# ── Build full sequences (prompt + completion) with completion-only labels ────
sft_input_ids_list = []
sft_labels_list    = []

for ex in sft_examples:
    chat = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": ex["attacker_turns"][0]},
    ]
    prompt_str     = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    completion_str = ex["gold_completion"]

    # Tokenise prompt and full sequence separately to find split point
    prompt_ids = tokenizer.encode(prompt_str,                    add_special_tokens=False)
    full_ids   = tokenizer.encode(prompt_str + completion_str,   add_special_tokens=False)
    full_ids   = full_ids[:SFT_MAX_LENGTH]

    # Mask prompt tokens with -100 so loss is only on completion
    prompt_len = min(len(prompt_ids), len(full_ids))
    labels     = [-100] * prompt_len + full_ids[prompt_len:]

    sft_input_ids_list.append(full_ids)
    sft_labels_list.append(labels)

# ── Pad batch to uniform length ───────────────────────────────────────────────
pad_id  = tokenizer.pad_token_id
max_len = max(len(ids) for ids in sft_input_ids_list)

def pad_to(seq, length, pad_value):
    return seq + [pad_value] * (length - len(seq))

input_ids_tensor = torch.tensor(
    [pad_to(ids, max_len, pad_id) for ids in sft_input_ids_list],
    dtype=torch.long,
)
labels_tensor = torch.tensor(
    [pad_to(lbl, max_len, -100) for lbl in sft_labels_list],
    dtype=torch.long,
)
attention_mask = (input_ids_tensor != pad_id).long()

print(f"SFT batch shape: {input_ids_tensor.shape}  (examples \u00d7 tokens, padded to {max_len})")

# ── Warm-start training loop ──────────────────────────────────────────────────
model.train()
optimizer = AdamW(model.parameters(), lr=SFT_LR)

input_ids_tensor = input_ids_tensor.to(DEVICE)
labels_tensor    = labels_tensor.to(DEVICE)
attention_mask   = attention_mask.to(DEVICE)

print(f"Running SFT warm-start for {SFT_EPOCHS} epoch(s) on {len(sft_examples)} gold examples...")

for epoch in range(SFT_EPOCHS):
    optimizer.zero_grad()
    outputs = model(
        input_ids=input_ids_tensor,
        attention_mask=attention_mask,
        labels=labels_tensor,
    )
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f"  [SFT epoch {epoch + 1}/{SFT_EPOCHS}] loss = {loss.item():.4f}")

# Clean up — GRPO will create its own optimizer
del optimizer
if DEVICE == "cuda":
    torch.cuda.empty_cache()

model.eval()
print("\u2705 SFT warm-start complete. Model is ready for GRPO.")

## Cell 7 — Build GRPO Dataset

In [ ]:
# Dataset sized to step budget — acts as secondary stop safeguard
n_samples = NUM_STEPS * BATCH_SIZE + 10
print(f"Building dataset ({n_samples} samples for {NUM_STEPS} steps)...")
dataset = build_training_dataset(env, tokenizer, n_samples=n_samples)
print(f"\u2705 Dataset built. Sample prompt preview:")
print(dataset[0]["prompt"][:200], "...")

## Cell 8 — GRPO Training

In [ ]:
config = GRPOConfig(
    output_dir                  = OUTPUT_DIR,
    max_steps                   = NUM_STEPS,
    per_device_train_batch_size = BATCH_SIZE,
    num_generations             = NUM_GENERATIONS,
    logging_steps               = LOGGING_STEPS,
    save_steps                  = max(NUM_STEPS // 2, 1),
    max_completion_length       = 128,
    max_prompt_length           = 512,
    learning_rate               = 5e-7,
    beta                        = 0.04,
    temperature                 = 0.9,
    lr_scheduler_type           = "constant",
    bf16                        = HAS_GPU,
    use_cpu                     = not HAS_GPU,
    report_to                   = "none",
)

trainer = GRPOTrainer(
    model            = model,
    args             = config,
    reward_funcs     = [real_reward_fn],
    train_dataset    = dataset,
    processing_class = tokenizer,
    callbacks        = [TrainingCallback(max_steps=NUM_STEPS)],
)

print(f"--- TRUSTSHIELD {PHASE.upper()} GRPO TRAINING ({NUM_STEPS} STEPS) ---")
print(f"LR: 5e-7 | Beta: 0.04 | Temp: 0.9 | Device: {DEVICE}")
print("Starting...")

trainer.train()

print("\u2705 GRPO training complete.")

## Cell 9 — Save Results, Plot Curves & Push to HF Space

In [ ]:
log_history = trainer.state.log_history

# Save training curves
save_training_curves(log_history, RESULTS_DIR)

# Save JSON log
log_path = os.path.join(RESULTS_DIR, f"training_log_{PHASE}.json")
with open(log_path, "w") as f:
    json.dump(log_history, f, indent=2)
print(f"\u2705 Saved training log \u2192 {log_path}")

# Push to HF Space
if hf_token:
    push_to_space(hf_token)
else:
    print("\u26a0\ufe0f  No HF_TOKEN — skipping push. Results saved locally only.")

print("\u2705 All done.")